In [25]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv("HousePricePrediction.csv")

In [27]:
df.head()

,Id,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,0,60,RL,8450,Inside,1Fam,5,2003,2003,VinylSd,0.0,856.0,208500.0
1,1,20,RL,9600,FR2,1Fam,8,1976,1976,MetalSd,0.0,1262.0,181500.0
2,2,60,RL,11250,Inside,1Fam,5,2001,2002,VinylSd,0.0,920.0,223500.0
3,3,70,RL,9550,Corner,1Fam,5,1915,1970,Wd Sdng,0.0,756.0,140000.0
4,4,60,RL,14260,FR2,1Fam,5,2000,2000,VinylSd,0.0,1145.0,250000.0


In [28]:
df.shape

(2919, 13)

In [29]:
df.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           4
LotArea            0
LotConfig          0
BldgType           0
OverallCond        0
YearBuilt          0
YearRemodAdd       0
Exterior1st        1
BsmtFinSF2         1
TotalBsmtSF        1
SalePrice       1459
dtype: int64

In [30]:
# df = df.dropna()

# ── Split train vs test rows ──────────────────────────
# Rows where SalePrice is NaN are test rows (no label)
train_df = df[df['SalePrice'].notna()].copy()
test_df  = df[df['SalePrice'].isna()].copy()

print("Train rows:", train_df.shape)
print("Test rows :", test_df.shape)


Train rows: (1460, 13)
Test rows : (1459, 13)


In [31]:

# ── Drop irrelevant columns ───────────────────────────
# Id = just a row number
train_df.drop(columns=['Id'], inplace=True)

print("Columns remaining:", train_df.columns.tolist())

Columns remaining: ['MSSubClass', 'MSZoning', 'LotArea', 'LotConfig', 'BldgType', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'Exterior1st', 'BsmtFinSF2', 'TotalBsmtSF', 'SalePrice']


In [33]:
# ── Fill Missing Values ───────────────────────────────





# Numerical columns → fill with median
num_cols = train_df.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols.remove('SalePrice')  # we do not need target

for col in num_cols:
    median_val = train_df[col].median()
    train_df[col].fillna(median_val, inplace=True)
    print(f"{col}: filled with median = {median_val}")






# Categorical columns → fill with mode
cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    mode_val = train_df[col].mode()[0]
    train_df[col].fillna(mode_val, inplace=True)
    print(f"{col}: filled with mode = {mode_val}")




# finally check no missing values remain
print("\nMissing values left:", train_df.isnull().sum().sum())

MSSubClass: filled with median = 50.0
LotArea: filled with median = 9478.5
OverallCond: filled with median = 5.0
YearBuilt: filled with median = 1973.0
YearRemodAdd: filled with median = 1994.0
BsmtFinSF2: filled with median = 0.0
TotalBsmtSF: filled with median = 991.5
MSZoning: filled with mode = RL
LotConfig: filled with mode = Inside
BldgType: filled with mode = 1Fam
Exterior1st: filled with mode = VinylSd

Missing values left: 0


C:\Users\Madhav\AppData\Local\Temp\ipykernel_10416\2239808807.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(median_val, inplace=True)
C:\Users\Madhav\AppData\Local\Temp\ipykernel_10416\2239808807.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [34]:
# ── Encode Categorical Columns ────────────────────────

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    train_df[col] = le.fit_transform(train_df[col])
    print(f"{col}: encoded ✅")

train_df.head()

MSZoning: encoded ✅
LotConfig: encoded ✅
BldgType: encoded ✅
Exterior1st: encoded ✅


,MSSubClass,MSZoning,LotArea,LotConfig,BldgType,OverallCond,YearBuilt,YearRemodAdd,Exterior1st,BsmtFinSF2,TotalBsmtSF,SalePrice
0,60,3,8450,4,0,5,2003,2003,12,0.0,856.0,208500.0
1,20,3,9600,2,0,8,1976,1976,8,0.0,1262.0,181500.0
2,60,3,11250,4,0,5,2001,2002,12,0.0,920.0,223500.0
3,70,3,9550,0,0,5,1915,1970,13,0.0,756.0,140000.0
4,60,3,14260,2,0,5,2000,2000,12,0.0,1145.0,250000.0


In [35]:
# ── Select Features ───────────────────────────────────
X = train_df.drop('SalePrice', axis=True)
y = train_df['SalePrice']

print("\nX shape:", X.shape)
print("y shape:", y.shape)


X shape: (1460, 11)
y shape: (1460,)


In [36]:
# ── Scale Numerical Features ──────────────────────────
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaling done ✅")


Scaling done ✅


In [13]:
from sklearn.model_selection import train_test_split

# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# ── Train Test Split ──────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # so results are reproducible
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [14]:
from sklearn.linear_model import LinearRegression

regr = LinearRegression()

In [15]:
regr.fit(X_train, y_train)

y_pred = regr.predict(X_test)

regr.score(X_test, y_test)  # returns R² score by default

0.5750904367788137

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_true=y_test, y_pred=y_pred)
mse = mean_squared_error(y_true=y_test, y_pred=y_pred)

print("MAE:", mae)
print("MSE:", mse)

MAE: 37871.4288508465
MSE: 3259194958.459901


In [17]:
from sklearn.metrics import r2_score

y_pred = regr.predict(X_test)
r2_score = r2_score(y_test, y_pred)
print(r2_score)

0.5750904367788137
